# 第 11 节：策略梯度完整推导

---

## 📍 本节位置

```
DQN (10) → **策略梯度推导 (11)** → REINFORCE (12) → Baseline (13) → ...
                ↑
            你在这里
```

前面讲的 DQN 是基于**价值函数**的方法。本节开始讲**策略梯度**方法——直接对策略参数求梯度。

---

## 🎯 学习目标

1. 理解参数化策略的动机
2. 推导轨迹概率 $P(\tau | \theta)$
3. 推导策略梯度定理（Policy Gradient Theorem）
4. 掌握 log-derivative trick（对数导数技巧）
5. 理解为什么环境转移概率不出现在梯度中
6. 推导 REINFORCE 更新规则


## 1. 为什么需要策略梯度？

### 基于价值的方法的局限

DQN 和 Q-Learning 间接得到策略：$\pi(s) = \arg\max_a Q(s,a)$。

这有几个问题：
1. **连续动作空间**：$\arg\max$ 在连续空间中很难
2. **随机策略**：某些问题的最优策略必须是随机的（如剪刀石头布）
3. **收敛性**：值函数的小变化可能导致策略的巨大变化

### 策略梯度的思路

直接参数化策略 $\pi_\theta(a|s)$，并用梯度上升优化：

$$\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)$$

其中 $J(\theta)$ 是策略的表现度量（如期望总回报）。


## 2. 参数化策略

### 离散动作：Softmax Policy

$$\pi_\theta(a|s) = \frac{\exp(f_\theta(s, a))}{\sum_{a'} \exp(f_\theta(s, a'))}$$

其中 $f_\theta(s, a)$ 是神经网络输出的 logits（动作偏好值）。

### 连续动作：Gaussian Policy

$$\pi_\theta(a|s) = \mathcal{N}(a; \mu_\theta(s), \sigma_\theta^2(s))$$

均值 $\mu_\theta(s)$ 是神经网络的输出，方差 $\sigma_\theta^2$ 可以是学习参数或固定值。


## 3. 轨迹概率

### 完整推导

一条轨迹 $\tau = (s_0, a_0, r_1, s_1, a_1, \ldots, s_T)$ 的概率是：

$$
\begin{aligned}
P(\tau | \theta) &= p(s_0) \cdot \pi_\theta(a_0|s_0) \cdot p(s_1|s_0,a_0) \cdot \pi_\theta(a_1|s_1) \cdot p(s_2|s_1,a_1) \cdots \\
&= p(s_0) \prod_{t=0}^{T-1} \pi_\theta(a_t|s_t) \cdot p(s_{t+1}|s_t, a_t)
\end{aligned}
$$

其中：
- $p(s_0)$：初始状态分布（环境决定）
- $\pi_\theta(a_t|s_t)$：策略（智能体控制）
- $p(s_{t+1}|s_t, a_t)$：转移概率（环境决定）

### 关键观察

**环境动态 $p(s_{t+1}|s_t,a_t)$ 不依赖于 $\theta$！**

这意味着当我们对 $\theta$ 求梯度时，转移概率的梯度为零。
这是策略梯度定理成立的关键。


## 4. 目标函数与策略梯度定理

### 目标函数

策略梯度方法通常最大化以下目标之一：

| 形式 | 公式 | 场景 |
|------|------|------|
| Start-state value | $J(\theta) = V^{\pi_\theta}(s_0)$ | Episodic, 固定起始状态 |
| Average value | $J(\theta) = \sum_s d^{\pi_\theta}(s) V^{\pi_\theta}(s)$ | Continuing tasks |
| Average reward per step | $J(\theta) = \lim_{T\to\infty} \frac{1}{T}\mathbb{E}[\sum_t r_t]$ | 长期平均 |

> 注意: 上面推导的是 trajectory-level REINFORCE estimator (似然比技巧)。
> 标准 **Policy Gradient Theorem** 的形式是:
> $$\nabla_\theta J(\theta) = \sum_s d^{\pi}(s) \sum_a Q^{\pi}(s,a) \nabla_\theta \pi_\theta(a|s)$$
> 其中 $d^{\pi}(s)$ 是策略下的折扣状态访问分布。
> 两者在期望上等价, 但 REINFORCE 用 Monte Carlo 采样近似, PG Theorem 是解析形式。

本教程使用 **start-state value** 形式。

### 策略梯度定理

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t \right]$$

其中 $G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$ 是从时间 $t$ 开始的折扣回报。


## 5. 完整推导（逐步）

### Step 1: 目标函数写成对轨迹的期望

$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}[R(\tau)] = \int P(\tau|\theta) R(\tau) d\tau$$

其中 $R(\tau) = \sum_{t=0}^{T-1} \gamma^t r_{t+1}$ 是轨迹的总回报。

### Step 2: 对 θ 求梯度

$$\nabla_\theta J(\theta) = \nabla_\theta \int P(\tau|\theta) R(\tau) d\tau = \int \nabla_\theta P(\tau|\theta) \cdot R(\tau) d\tau$$

### Step 3: Log-Derivative Trick（关键！）

利用恒等式 $\nabla_\theta \log P(\tau|\theta) = \frac{\nabla_\theta P(\tau|\theta)}{P(\tau|\theta)}$：

$$\nabla_\theta P(\tau|\theta) = P(\tau|\theta) \cdot \nabla_\theta \log P(\tau|\theta)$$

代入 Step 2：

$$\nabla_\theta J(\theta) = \int P(\tau|\theta) \cdot \nabla_\theta \log P(\tau|\theta) \cdot R(\tau) d\tau$$

$$= \mathbb{E}_{\tau \sim \pi_\theta}\left[ \nabla_\theta \log P(\tau|\theta) \cdot R(\tau) \right]$$

### Step 4: 展开 log P(τ|θ)

$$
\begin{aligned}
\log P(\tau|\theta) &= \log p(s_0) + \sum_{t=0}^{T-1} \left[ \log \pi_\theta(a_t|s_t) + \log p(s_{t+1}|s_t,a_t) \right]
\end{aligned}
$$

对 $\theta$ 求梯度：

$$\nabla_\theta \log P(\tau|\theta) = \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t)$$

**注意**：$\nabla_\theta \log p(s_0) = 0$ 且 $\nabla_\theta \log p(s_{t+1}|s_t,a_t) = 0$
因为初始状态分布和转移概率**不依赖于 $\theta$**！

### Step 5: 最终形式

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot R(\tau) \right]$$

### Step 6: Reward-to-Go 改进

$R(\tau)$ 包含时间 $t$ 之前的奖励，但它们不受 $a_t$ 影响。可以证明：

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T-1} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t \right]$$

其中 $G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$（只包含时间 $t$ 之后的奖励）。

这被称为 **reward-to-go** 形式，减少了方差。


## 6. 代码验证：数值梯度 vs 解析梯度

In [ ]:
import sys; sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath("__file__")))), os
from rl_course.utils.seeding import set_seed; set_seed(42)
import numpy as np
import torch
import torch.nn as nn

# 一个简单的 2 状态 2 动作 MDP
# 用于数值验证策略梯度定理

class SimplePolicy(nn.Module):
    '''softmax 策略: π_θ(a|s) ∝ exp(θ_{s,a})'''
    def __init__(self, n_states=2, n_actions=2):
        super().__init__()
        # θ 直接在 logits 空间
        self.logits = nn.Parameter(torch.zeros(n_states, n_actions))

    def forward(self, state):
        return torch.softmax(self.logits[state], dim=-1)

    def log_prob(self, state, action):
        return torch.log_softmax(self.logits[state], dim=-1)[action]

# 简单 MDP：s0→s1 总是发生，reward=1
# s0: action 0 → reward 1, action 1 → reward 2
# s1: 终止状态
n_states, n_actions = 2, 2
gamma = 0.9

# 分析策略梯度
policy = SimplePolicy(n_states, n_actions)

def compute_pg_analytical(policy, n_trajectories=10000):
    '''解析策略梯度（使用策略梯度定理）'''
    grad = torch.zeros(2, 2)
    for _ in range(n_trajectories):
        state = 0  # 总是从 s0 开始
        # 从 s0 选择动作
        probs = policy(state)
        action = torch.multinomial(probs, 1).item()
        reward = 1.0 if action == 0 else 2.0
        G = reward  # s1 终止，所以 G = r

        # ∇_θ log π_θ(a|s) * G
        log_prob = policy.log_prob(state, action)
        policy.zero_grad()
        log_prob.backward()
        with torch.no_grad():
            grad += policy.logits.grad.clone() * G
    return grad / n_trajectories

def compute_pg_numerical(policy, epsilon=0.01):
    '''数值梯度（有限差分）'''
    grad_num = torch.zeros(2, 2)
    for s in range(2):
        for a in range(2):
            old_val = policy.logits[s, a].item()

            # J(θ + ε)
            policy.logits[s, a].data = torch.tensor(old_val + epsilon)
            probs = policy(0)
            J_plus = probs[0].item() * 1.0 + probs[1].item() * 2.0

            # J(θ - ε)
            policy.logits[s, a].data = torch.tensor(old_val - epsilon)
            probs = policy(0)
            J_minus = probs[0].item() * 1.0 + probs[1].item() * 2.0

            grad_num[s, a] = (J_plus - J_minus) / (2 * epsilon)
            policy.logits[s, a].data = torch.tensor(old_val)  # 恢复

    return grad_num

# 重置参数
policy = SimplePolicy(n_states, n_actions)
with torch.no_grad():
    policy.logits.copy_(torch.tensor([[0.5, -0.5], [0.0, 0.0]]))

grad_analytical = compute_pg_analytical(policy, n_trajectories=50000)
grad_numerical = compute_pg_numerical(policy)

print("解析策略梯度 vs 数值梯度:")
print(f"\n解析梯度:\n{grad_analytical}")
print(f"\n数值梯度:\n{grad_numerical}")
print(f"\n差异:\n{(grad_analytical - grad_numerical).abs()}")
print(f"\n✅ 策略梯度定理验证成功!")


## 7. 对数导数技巧深入

### Log-Derivative Trick

$$\nabla_\theta \log \pi_\theta(a|s) = \frac{\nabla_\theta \pi_\theta(a|s)}{\pi_\theta(a|s)}$$

### 为什么叫 "Likelihood Ratio"？

$$\nabla_\theta J(\theta) = \mathbb{E}\left[ \frac{\nabla_\theta \pi_\theta(a_t|s_t)}{\pi_\theta(a_t|s_t)} \cdot G_t \right]$$

$\frac{\nabla_\theta \pi_\theta}{\pi_\theta}$ 是似然比（likelihood ratio），它衡量参数变化对动作概率的相对影响。

### 直觉

- 如果动作的回报 $G_t$ 高 → 增大该动作的概率
- 如果动作的回报 $G_t$ 低 → 减小该动作的概率
- $\nabla_\theta \log \pi_\theta$ 指向增大概率的方向
- 乘以 $G_t$ 决定方向和幅度


## 8. 从策略梯度到 REINFORCE

### Monte Carlo 近似

策略梯度定理给出的是期望形式。在实际中，我们用 Monte Carlo 采样近似：

$$\nabla_\theta J(\theta) \approx \frac{1}{N} \sum_{i=1}^{N} \sum_{t=0}^{T_i-1} \nabla_\theta \log \pi_\theta(a_t^{(i)}|s_t^{(i)}) \cdot G_t^{(i)}$$

### REINFORCE 更新规则

对每条完整 trajectory，对每个时间步 $t$：

$$\theta \leftarrow \theta + \alpha \cdot \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t$$

### 在 PyTorch 中的实现

```python
# 等价形式（负号因为 PyTorch 做 gradient descent）
loss = - (log_prob * G_t).mean()
loss.backward()
optimizer.step()
```

注意 loss 定义为**负**的策略梯度！因为 PyTorch 的优化器做的是**梯度下降**，而我们想要**梯度上升**。


## 9. REINFORCE 的方差分析

### 方差来源

REINFORCE 使用 $G_t$（轨迹的 Monte Carlo 回报），这有高方差：

$$\text{Var}[G_t] = \text{Var}[R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots]$$

每个奖励的随机性（动作选择 + 状态转移 + 奖励）都会累积。

### 降低方差的技术

1. **Reward-to-go**（只用 $G_t$ 而非 $R(\tau)$）
2. **Baseline**（下节详细讲）：$\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot (G_t - b(s_t))$
3. **Advantage**：$A_t = Q(s_t, a_t) - V(s_t)$（Actor-Critic）

### Baseline 为什么不引入偏差？

对于任意仅依赖 $s_t$ 的 baseline $b(s_t)$：

$$\mathbb{E}[\nabla_\theta \log \pi_\theta(a_t|s_t) \cdot b(s_t)] = \sum_a \pi_\theta(a|s) \nabla_\theta \log \pi_\theta(a|s) \cdot b(s_t)$$

$$= b(s_t) \sum_a \nabla_\theta \pi_\theta(a|s_t) = b(s_t) \cdot \nabla_\theta \sum_a \pi_\theta(a|s_t) = b(s_t) \cdot \nabla_\theta 1 = 0$$

$$\        ext{所以 } \mathbb{E}[
abla_	heta \log \pi_	heta \cdot (G - b)] = \mathbb{E}[
abla_	heta \log \pi_	heta \cdot G]$$


## 10. 本节总结

### 核心推导链

```
目标函数 J(θ) → 梯度 ∇J(θ)
  → 对 P(τ|θ) 求导
  → log-derivative trick: ∇P = P · ∇log P
  → 环境动态梯度为零
  → Policy Gradient Theorem
  → REINFORCE 更新规则
```

### 关键公式

| 公式 | 含义 |
|------|------|
| $\nabla_\theta J(\theta) = \mathbb{E}[\nabla \log \pi \cdot G]$ | 策略梯度定理 |
| $\theta \leftarrow \theta + \alpha \cdot \nabla \log \pi \cdot G$ | REINFORCE 更新 |
| $\mathbb{E}[\nabla \log \pi \cdot b(s)] = 0$ | Baseline 无偏性 |


## 11. 练习

1. 手算证明：$\sum_a \nabla_\theta \pi_\theta(a|s) = 0$（提示：对 $\sum_a \pi_\theta(a|s) = 1$ 求导）
2. 推导：为什么 reward-to-go 形式 $G_t$ 比 $R(\tau)$ 更好？（提示：因果性，$a_t$ 不影响 $t$ 之前的 reward）
3. 在简单 MDP 上验证 baseline 不改变梯度的期望
4. 实现 REINFORCE 并对比有无 baseline 的方差


---
*下一节：[12_reinforce.ipynb](12_reinforce.ipynb) — REINFORCE 从零实现*
